#Import libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


##Data ingestions 
###Understand the four tables

In [0]:
#loading the customers table from Databricks into a Spark DataFrame and then store it in the variable customer
#Spark:Conceptually, it functions exactly like a table in a relational database or a spreadsheet, but with one massive difference: while a regular spreadsheet sits on a single computer, a Spark DataFrame is split across a cluster of thousands of computers, allowing it to process massive big data workloads in parallel.
customer=spark.table("workspace.default.customers")
customers=customer.toPandas()
display(customers)




In [0]:
#loading the orders table from Databricks into a Spark DataFrame and then store it in the variable orders
orders=spark.table("workspace.default.orders")
orders=orders.toPandas()
display(orders)

In [0]:
#loading the payments table from Databricks into a Spark DataFrame and then store it in the variable payments
payments=spark.table("workspace.default.payments")
payments=payments.toPandas()
display(payments)

In [0]:
#loading the products table from Databricks into a Spark DataFrame and then store it in the variable products
products=spark.table("workspace.default.products")
products=products.toPandas()
display(products)

- Key findings: We have 4 levels of information
- Customers represents  the customer Customer ID, AGE, signupdate, city and segment 
- Orders represents the 
- Payments
- Products

#Exploratory Data analysis

In [0]:
#The .shape tells us the dimensions of the table we have 10000 rows and 5 columns for the customers table. 
customers.shape


In [0]:
#The .shape tells us that the we have 50120 rows, and 8 columns for the orders table
orders.shape


In [0]:
#The .shape tells us that we have 50000 rows and 4 columns for the payments table
payments.shape



In [0]:
#The .shape tells us that we have 20 rows and 4 columns for the products table
products.shape


In [0]:
#check the columns found in the customers table and help state the data types for each column
#check what data fields are available before you perform data analysis, filtering, or calculations.
customers.columns

In [0]:
#Check the columns found in the orders table 
orders.columns

In [0]:
#Check the columns found in the products table 
products.columns

In [0]:
#Check the columns found in the payments table
payments.columns

In [0]:
#structural summary of a DataFrame from the customers table we see that the customerID is in the correct formating as an integer
#Age is reflecting the correct data type as a float64 ! this is flagged as age cannot be a float
#City is a string object works well
#SignupDate is a date object we can convert it to date and time 
#CustomerSegment is a string object we can make it a category object
customers.info()

In [0]:
#Structural summary of DataFrame from the orders table
#ordersID is in the correct data type as int64
#CustomerID is in the correct data type as int64
#Orderdate is an object and might need to be converted to a data
#ProductID is in the correct format
#Quantity is in the correct format
#Discount is in the correct format
#Payment Method might need to be converted to string but will decide
#Status is an object
orders.info()

In [0]:
#PaymentsID is in the correct format
#OrderID is in the correct format
#PaymentDate is an object and might need to be converted to a date
#PaymentStatus is an object

payments.info()

In [0]:
#Products is in the correct data type
#Products is in the correct data type
#Catergory we need to decide whether we need to convert it.
#UnitPrice is in the correct data type
products.info()
#check the data types for the products table


#Check for data Quality

- Outcome for this section 
- Find missing value
- Incorrect data types(we really looking into this)
- Invalid values
- Inconsistent categories 


In [0]:
#Number of nulls in the customers table
#There is 180 nulls in the Age columns and 119 in the City columns 
customers.isnull().sum()

In [0]:
#Number of nulls that are duplicated in the customers table
#No duplicates in the customers table
customers.duplicated().sum()

In [0]:
#statistical summary of a dataset in the customers data

customers.describe()

In [0]:
#OrderDate has 35 null value 
#Quantity has 80 null values
#Discount has 221 Null value
#PaymentMethod 452
#Data type int64
orders.isnull().sum()

In [0]:
#We have 120 duplicated values
orders.duplicated().sum()

In [0]:
orders.describe().round(2)
#Pandas is used for data manipulation and analysis so below is a satistical summary of the orders table

In [0]:
#No null 
products.isnull().sum()

In [0]:
#There is no duplicated values in the columns or rows
products.duplicated().sum()

In [0]:
#For me it made sense to use the statistical representation of the unit price then the other columns 
products.describe()["UnitPrice"].round(2)

In [0]:
#PaymentDate date has 35 null values
payments.isnull().sum()

In [0]:
#No duplicated data found in this table
payments.duplicated().sum()


In [0]:
#No need for the Describe, this table is used 

#EDA Analyse individual variable

In [0]:
customers.columns


In [0]:

customers["City"].value_counts()

In [0]:
#Find the inconsistent values of city
customers["City"].value_counts(dropna=False)

In [0]:
#Trying to Upper case the city, and remove any unwanted spaces and correct spelling. Replace NA with "unknown"
customers["City"] = (
    customers["City"]
    .str.strip()
    .str.title()
    .replace({"Mashad": "Mashhad"})
)
customers["City"] = customers["City"].fillna("Unknown")

In [0]:
#Getting a summary of the  data
customers["City"].value_counts(dropna=False)

In [0]:
#Different number of segment, there are 3 catergories of customersegment namely Regular(5563), New(3462), and Returning(975)
customers["CustomerSegment"].value_counts()


In [0]:

# For the age column fill the 180 blanks with the median, then store as whole numbers
customers["Age"] = customers["Age"].fillna(customers["Age"].median())
customers["Age"] = customers["Age"].astype("Int64")


In [0]:
#Correcting the date data type to datetime datatype 
customers["SignupDate"] = pd.to_datetime(customers["SignupDate"])


In [0]:
#The columns shown here for better overview of the data
orders.columns



In [0]:
# There are 3 types status in the orders table namely completed, cancelled, Returned
#Completed: 46105
#Cancelled: 2477
#Returned: 1538
#NaN:0
orders["Status"].value_counts(dropna=False)


In [0]:
#Getting an overview of the discounts added(dropna=False) to help us count the number of empty values
orders["Discount"].value_counts(dropna=False)

In [0]:
# Remove unwanted spaces, standardise capitalisation, and label blanks "Unknown" so we can count them
orders["PaymentMethod"] = orders["PaymentMethod"].str.strip().str.title()
orders["PaymentMethod"] = orders["PaymentMethod"].replace({"Cardtocard": "CardToCard"})
orders["PaymentMethod"] = orders["PaymentMethod"].fillna("Unknown")

In [0]:
orders["PaymentMethod"].value_counts(dropna=False)

In [0]:
#We found 449 blanks (after cleaning). We decided to labelled them “Unknown” and standardised capitalisation, restoring CardToCard. Reasons:This keeps those orders in the totals, and lets us see whether “Unknown” payments fail more often.
orders["PaymentMethod"].value_counts(dropna=False)

In [0]:
orders["Discount"] = orders["Discount"].fillna(0)
orders["Discount"].value_counts(dropna=False).sort_index()

In [0]:
#Working on the orders table duplicates:120 
orders.duplicated().sum() 

In [0]:
#Drop the duplicates
orders = orders.drop_duplicates()


In [0]:
#Check whether there are any remaining duplicates per OrderID column
orders["OrderID"].is_unique 

In [0]:
#Convert the orderDate to the correct Data type
orders["OrderDate"] = pd.to_datetime(orders["OrderDate"])
orders["OrderDate"].isna().sum()   # 35
orders["OrderDate"].min(), orders["OrderDate"].max()

In [0]:
#We picked up on negative quanitiy we needed to confirm whether we move forward or drop, we will keep them for now but we need to exclude them from the revenue (negative numbers are errors)
orders["Quantity"].value_counts(dropna=False).sort_index()

In [0]:
#We have decided to drop the missing quantity 
before = len(orders)
orders = orders.dropna(subset=["Quantity"])
print(before, "->", len(orders))  

#Convert Column it back to whole numbers 
orders["Quantity"] = orders["Quantity"].astype(int)

#found 80 blank quantities. Decided to remove those rows. Why: revenue can’t be calculated without a quantity, so the effect on totals is tiny. One honest side effect is that those orders also drop out of the order counts and cancellation rates.

In [0]:
#Fill the missing discounts with 0. 
orders["Discount"] = orders["Discount"].fillna(0)
orders["Discount"].value_counts().sort_index()

In [0]:
#Dropping the negative values
orders = orders[orders["Quantity"] > 0]
orders["Quantity"].value_counts().sort_index() 

In [0]:
#Quantity: found 80 blanks plus 25 rows with zero or negative quantities (-2, -1, 0). We decided have them all removed, along with the duplicates. The logic would be that a sale can’t have zero or negative units, and revenue can’t be calculated without a quantity. Totals are barely affected.

In [0]:
orders["Quantity"] = orders["Quantity"].astype(int)

In [0]:
payments["PaymentStatus"].value_counts(dropna=False)

#There are 3 types of payment status in the payments table namely completed, failed, and refunded
#We have 46569 completed payments,1924 Failed, and 1507 Refunded

In [0]:
#Number of blanks in the payments table. Payment dates has 35 blanks/Nulls. 
payments.isna().sum()

In [0]:
#Remove any unwanted spaces and Upper case the first letter
payments["PaymentStatus"] = payments["PaymentStatus"].astype(str).str.strip().str.title()
#Correct Date format
payments["PaymentDate"] = pd.to_datetime(payments["PaymentDate"])
#found 35 blank PaymentDates, no duplicates, three clean statuses (Paid, Failed, Refunded). We decided to have converted PaymentDate to a date and left the blanks until the merge. Those same 35 orders have no OrderDate either, so we decide on them once, in the merge.

In [0]:
products.columns

In [0]:
products

In [0]:
#Check for duplicates although looking at the table above there seems to be non.
products.isna().sum()          
products.duplicated().sum()   
products.dtypes

In [0]:
#Product category 6 categories. 
products["Category"].value_counts()


In [0]:
#Correct the caps for the ProductName and Category.Remove any unwanted spaces for both columns
#products["ProductName"] = products["ProductName"].str.strip().str.title()
products["Category"] = products["Category"].str.strip().str.title()

In [0]:
#This section is to verify the prices. We found that Monitor listed at 15, which is cheaper than a Wireless Mouse (0) plus a USB-C cable (2) and low for a monitor. We kept the price as given and flagged it as possibly incorrect. The orders are real sales, and I have no source to confirm the correct price. Monitor is about 0.66% of revenue (687 orders), so the effect on totals is small. If the price is wrong, Monitor revenue and the Electronics category are understated
products.sort_values("UnitPrice")[["ProductName", "Category", "UnitPrice"]]


#Merge tables

Used the below to find the matching columns, then merge them all into one table where possible
- -The customers table has a matching column(common column) through CustomerID to the orders table.
- -The orders table has a matching column(common column) through OrderID to the payments table.
- -The orders table has a matching column(common column) through ProductID to the products table.
- -The Orders table is the Main charactor holding the common columns OrderID, CustomerID and ProductID

Merge type: Left joins from orders, so every order line is kept even if a match is missing.
Check: the row count stayed at 49,895 after each join, so no rows were duplicated or lost.

In [0]:
#Split the merge of each table to ensure the consistant row count:
print("orders:", len(orders))                                   

step1 = orders.merge(payments, on="OrderID", how="left")
print("after payments:", len(step1))                            

step2 = step1.merge(products, on="ProductID", how="left")
print("after products:", len(step2))                            

final_grouped_table = step2.merge(customers, on="CustomerID", how="left")
print("after customers:", len(final_grouped_table))             # 49,895

In [0]:
final_grouped_table.isna().sum()

In [0]:
final_grouped_table["City"] = final_grouped_table["City"].fillna("Unknown")
final_grouped_table["CustomerSegment"] = final_grouped_table["CustomerSegment"].fillna("Unknown")

2. Revenue and the revenue rule

In [0]:
#Revenue the revenue is 42,754 Completed payments. Cancelled, Returned, Failed and Refunded are not counted(false)
final_grouped_table["Revenue"] = (
    final_grouped_table["Quantity"]
    * final_grouped_table["UnitPrice"]
    * (1 - final_grouped_table["Discount"])
)

final_grouped_table["CountsAsRevenue"] = (
    (final_grouped_table["Status"] == "Completed")
    & (final_grouped_table["PaymentStatus"] == "Paid")
)
final_grouped_table["CountsAsRevenue"].value_counts()

3. Year and Month

In [0]:
final_grouped_table["OrderDate"] = pd.to_datetime(final_grouped_table["OrderDate"])
final_grouped_table["Year"] = final_grouped_table["OrderDate"].dt.year
final_grouped_table["MonthStart"] = final_grouped_table["OrderDate"].dt.to_period("M").dt.to_timestamp()

4. Headline numbers

In [0]:
revenue_df = final_grouped_table[final_grouped_table["CountsAsRevenue"]].copy()
revenue_df["Revenue"].sum().round(), len(revenue_df), revenue_df["Revenue"].mean().round(2)

#Charts and Graphs

Chart 1: headline numbers

In [0]:
#Our key performance indicators.
kpis = {
    "Total revenue": round(revenue_df["Revenue"].sum()),
    "Orders": len(revenue_df),
    "Average order value": round(revenue_df["Revenue"].mean(), 2),
}
kpis

Chart 2: is revenue growing, shrinking or flat?

In [0]:

monthly = (
    revenue_df.groupby("MonthStart")["Revenue"]
    .sum()
    .reset_index()
)
monthly.shape

In [0]:

# Below graph shows that the gradual as per the business case. The revenue then drops just at the start of 2026
plt.figure(figsize=(11, 5))
plt.plot(monthly["MonthStart"], monthly["Revenue"], marker="o")
plt.title("Monthly revenue, Jan 2024 to Jun 2026")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# 1. Did the number of counted orders fall?
#Between 2025-11-01 abd 2026-01-01 there was a constant drop
#Then from 2026-03-01 there was a increased number of orders, this represented the peak of purchases. Between 2026-04 and 2026-05-01 the changes were at a lower rate with a slight increase and decrease. We saw a drop from 2026-05-01 to 2026-06-01.
revenue_df.groupby("MonthStart")["OrderID"].count().tail(8)

In [0]:
# 2. Did the average order value fall?
revenue_df.groupby("MonthStart")["Revenue"].mean().round(2).tail(8)

In [0]:
# 3. Did the share of non-counted orders rise?
(final_grouped_table.groupby("MonthStart")["CountsAsRevenue"].mean() * 100).round(1).tail(8)

Revenue held steady at around 100,000 to 115,000 a month for two years, then fell by about a third from February 2026. The cause is fewer orders (about 1,500 a month down to about 1,000), not lower prices or more failed payments, so the first place to investigate is what changed in the shop in early February.

In [0]:
orders_per_month = revenue_df.groupby("MonthStart")["OrderID"].count().reset_index()
plt.figure(figsize=(11, 5))
plt.plot(orders_per_month["MonthStart"], orders_per_month["OrderID"], marker="o")
plt.title("Orders per month: a step down from February 2026")
plt.xlabel("Month"); plt.ylabel("Orders")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

chart 3: products and categories, revenue versus units sold

In [0]:
#The items which are best performing are not in the same category.
by_product = (
    revenue_df.groupby(["ProductName", "Category"])
    .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"))
    .reset_index()
)
by_product.sort_values("Revenue", ascending=False)

In [0]:
#Side by side comparison of the Revenue and Units by product
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

rev = by_product.sort_values("Revenue")
axes[0].barh(rev["ProductName"], rev["Revenue"])
axes[0].set_title("Revenue by product")
axes[0].set_xlabel("Revenue")

units = by_product.sort_values("Units")
axes[1].barh(units["ProductName"], units["Units"])
axes[1].set_title("Units sold by product")
axes[1].set_xlabel("Units")

plt.tight_layout()
plt.show()


The products that sell the most units are cheap items like notebooks and phone case, but they bring in little money. Revenue comes from higher-priced products such as headphones, office chairs and tablets, even though they sell far fewer units. So the shop’s income depends on a small number of expensive products, and that’s where stock and promotion effort should go

In [0]:
by_category = (
    revenue_df.groupby("Category")
    .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"))
    .sort_values("Revenue", ascending=False)
)
by_category

Chart 4: cities and customer segments

In [0]:
#Revenue and orders by city
by_city = (
    revenue_df.groupby("City")
    .agg(Revenue=("Revenue", "sum"), Orders=("OrderID", "count"))
    .sort_values("Revenue", ascending=False)
)
by_city



In [0]:
#Calculayions for the customer segments
by_segment = (
    revenue_df.groupby("CustomerSegment")
    .agg(Revenue=("Revenue", "sum"), Orders=("OrderID", "count"), Customers=("CustomerID", "nunique"))
)
by_segment["Revenue per customer"] = (by_segment["Revenue"] / by_segment["Customers"]).round(2)
by_segment

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cities: leave out Unknown so the ranking only shows real cities
city = by_city.drop(index="Unknown", errors="ignore").sort_values("Revenue")
axes[0].barh(city.index, city["Revenue"])
axes[0].set_title("Revenue by city")
axes[0].set_xlabel("Revenue")

# Segments: leave out Unknown (a single unregistered customer)
seg = by_segment.drop(index="Unknown", errors="ignore").sort_values("Revenue per customer", ascending=False)
axes[1].bar(seg.index, seg["Revenue per customer"])
axes[1].set_title("Revenue per customer by segment")
axes[1].set_ylabel("Revenue per customer")

plt.tight_layout()
plt.show()


Cities: Tehran leads by a wide margin, at about 823,000, Mashhad is next 376799.00, then Karaj and Isfahan at about 287865.50 each.

##Chart 6:Do bigger discounts lead to bigger orders, or just lower revenue?

1. Summary by discount level

In [0]:
by_discount = (
    revenue_df.groupby("Discount")
    .agg(
        Orders=("OrderID", "count"),
        Avg_units=("Quantity", "mean"),
        Avg_order_value=("Revenue", "mean"),
        Revenue=("Revenue", "sum"),
    )
    .round(2)
)
by_discount

2.Chart: average order value and units per order by discount

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(by_discount.index.astype(str), by_discount["Avg_order_value"])
axes[0].set_title("Average order value by discount level")
axes[0].set_xlabel("Discount")
axes[0].set_ylabel("Average order value")

axes[1].bar(by_discount.index.astype(str), by_discount["Avg_units"])
axes[1].set_title("Average units per order by discount level")
axes[1].set_xlabel("Discount")
axes[1].set_ylabel("Units per order")

plt.tight_layout()
plt.show()

3. How much money the discounts cost

In [0]:
gross = (revenue_df["Quantity"] * revenue_df["UnitPrice"]).sum()
discount_given = gross - revenue_df["Revenue"].sum()
round(gross), round(discount_given), round(discount_given / gross * 100, 1)

Customers buy about the same number of items whether the discount is 0% or 30%, so deeper discounts reduce revenue without growing the basket. The shop gave away about 246,000 in discounts, roughly 7.6% of gross sale. This shows basket size, not that the overall discount bring in more customers. VIP customers spend about the same as New and Regular ones (about 300 each), so the VIP label isn’t identifying higher-value customers.

#Recommendations

Investigate the February 2026 drop in orders. Monthly revenue fell from approximately 105,000 between January 2024 and January 2026 to around 71,000 from February to June 2026. Orders also dropped from roughly 1,500 to 1,000 per month, while average order value and payment success remained stable. Since the decline affected every city, customer segment, product category and payment method, investigate possible business-wide causes, such as reduced website traffic, changes in marketing, website updates or changes in data collection introduced in February.

Reduce reliance on deep discounts. Customers appear to purchase similar quantities at 30% off as they do without a discount. The 20% and 30% discount tiers accounted for approximately 87,000 in discounts given. Test limiting discounts to 10% or less and monitor whether order volumes remain stable while revenue improves.

Prioritise high-value products in stock and promotions. Headphones, Office Chairs and Tablets generate approximately 28% of revenue while accounting for less than 9% of units sold. In comparison, lower-priced, high-volume items such as notebooks, cables and phone cases sell more units but contribute less revenue per item. Focus stock planning and promotional efforts on higher-value products, and verify the Monitor’s listed price to ensure its revenue contribution is being calculated correctly.

4. Export for Looker Studio

In [0]:
export_df = final_grouped_table.copy()

export_df["OrderDate"] = export_df["OrderDate"].dt.strftime("%Y-%m-%d")
export_df["MonthStart"] = export_df["MonthStart"].dt.strftime("%Y-%m-%d")
export_df["PaymentDate"] = pd.to_datetime(export_df["PaymentDate"]).dt.strftime("%Y-%m-%d")
export_df["SignupDate"] = pd.to_datetime(export_df["SignupDate"]).dt.strftime("%Y-%m-%d")

export_df["Revenue"] = export_df["Revenue"].round(2)
export_df["CountedRevenue"] = export_df["Revenue"].where(export_df["CountsAsRevenue"], 0)
export_df["CountsAsRevenue"] = export_df["CountsAsRevenue"].map({True: "Yes", False: "No"})

export_df.shape

In [0]:
display(export_df)

In [0]:
export_df["Age"] = export_df["Age"].astype("float64")   # Spark can't read pandas' nullable integer type

spark.createDataFrame(export_df).write.mode("overwrite").saveAsTable("workspace.default.shop_joined")

In [0]:
spark.table("workspace.default.shop_joined").count()